# 1. Agent Service Control Panel and Run Configuration

In [ ]:
# ____________________________________________________________
# Purpose:
# Create a simple interactive input layer so the user can choose:
# - what kind of source data the agent is using
# - which dataset bundle to work from
# - how many synthetic patients to generate
# - which policy scenario to simulate
# - whether to include equity, risk, and governance checks
# ____________________________________________________________

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# ____________________________________________________________
# Create dropdown for source type
# - Represents the type of input the synthetic data service
#   is starting from.
# ____________________________________________________________
source_type = widgets.Dropdown(
    options=[
        "Public Population Data",
        "Metadata About a Population",
        "Structured Healthcare Dataset"
    ],
    value="Public Population Data",
    description="Source Type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px")
)

# ____________________________________________________________
# Create dropdown for dataset choice
# - Tells the agent what dataset bundle is being used
# ____________________________________________________________
source_dataset = widgets.Dropdown(
    options=[
        "Population Demographics + Chronic Disease + Healthcare Utilization",
        "Population Demographics Only",
        "Chronic Disease Dataset Only",
        "Healthcare Utilization Dataset Only"
    ],
    value="Population Demographics + Chronic Disease + Healthcare Utilization",
    description="Dataset:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

# ____________________________________________________________
# Create slider for synthetic population size
# - Controls how many synthetic patients will be generated.
# ____________________________________________________________
population_size = widgets.IntSlider(
    value=10000,
    min=1000,
    max=20000,
    step=1000,
    description="Synthetic Patients:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

# ____________________________________________________________
# Create dropdown for scenario selection
# - Allows the user to choose which policy simulation
#   they want the system to focus on.
# ____________________________________________________________
scenario_choice = widgets.Dropdown(
    options=[
        "Baseline",
        "Expanded Virtual Urgent Care",
        "Expanded Southlake@home + Outreach",
        "Equity-Aware Routing"
    ],
    value="Baseline",
    description="Scenario:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

# ____________________________________________________________
# Create optional feature checkboxes
# - Allow the agent to run with or without certain layers
# ____________________________________________________________
include_equity = widgets.Checkbox(
    value=True,
    description="Include equity/access profiling",
    indent=False
)

include_risk = widgets.Checkbox(
    value=True,
    description="Include predictive risk scoring",
    indent=False
)

include_governance = widgets.Checkbox(
    value=True,
    description="Run hygiene + governance checks",
    indent=False
)

# ____________________________________________________________
# Create run button
# - Acts like the “start service” button for the agent
# ____________________________________________________________
run_button = widgets.Button(
    description="Run Agent Service",
    button_style="success",
    icon="play"
)

# ____________________________________________________________
# Create an output area
# - This is where the chosen configuration will be displayed
# ____________________________________________________________
output = widgets.Output()

# ____________________________________________________________
# Global configuration dictionary
# - Stores the selected settings so later notebook cells
#   can reference them.
# ____________________________________________________________
agent_config = {}

# ____________________________________________________________
# Define what happens when the run button is clicked
# - Collects all widget values and stores them in agent_config
# ____________________________________________________________
def on_run_clicked(button):
    global agent_config

    with output:
        clear_output()

        # Save all selected control panel settings
        agent_config = {
            "source_type": source_type.value,
            "source_dataset": source_dataset.value,
            "population_size": population_size.value,
            "scenario_choice": scenario_choice.value,
            "include_equity": include_equity.value,
            "include_risk": include_risk.value,
            "include_governance": include_governance.value
        }

        # Display a readable summary for the user
        display(Markdown("## Agent Service Configuration"))
        for key, value in agent_config.items():
            print(f"{key}: {value}")

# Connect button click to function
run_button.on_click(on_run_clicked)

# ____________________________________________________________
# Display UI
# ____________________________________________________________
display(Markdown("## Synthetic Health Agent Control Panel"))
display(source_type)
display(source_dataset)
display(population_size)
display(scenario_choice)
display(include_equity)
display(include_risk)
display(include_governance)
display(run_button)
display(output)

# - turns notebook into interactive service rather than script
# - ipywidgets (create panel interface)
# - agent_config (all data is stored in this variable, controling pipline)


## Synthetic Health Agent Control Panel

Dropdown(description='Source Type:', layout=Layout(width='450px'), options=('Public Population Data', 'Metadat…

Dropdown(description='Dataset:', layout=Layout(width='650px'), options=('Population Demographics + Chronic Dis…

IntSlider(value=10000, description='Synthetic Patients:', layout=Layout(width='500px'), max=20000, min=1000, s…

Dropdown(description='Scenario:', layout=Layout(width='500px'), options=('Baseline', 'Expanded Virtual Urgent …

Checkbox(value=True, description='Include equity/access profiling', indent=False)

Checkbox(value=True, description='Include predictive risk scoring', indent=False)

Checkbox(value=True, description='Run hygiene + governance checks', indent=False)

Button(button_style='success', description='Run Agent Service', icon='play', style=ButtonStyle())

Output()

### What this section does
This step creates the **input layer for the synthetic health agent**.  
It allows the user to configure the system before running the rest of the notebook and stores those selections in an **`agent_config` dictionary**, which acts as the notebook’s runtime settings.  
  
### Why this step exists
The project is designed as an **agentic synthetic data service**, not just a static analysis script.  
This configuration layer makes the system **configurable and reusable**, allowing synthetic populations to be generated on demand.

### Technologies used
- **`ipywidgets`** for an interactive notebook interface  
- **Configuration dictionary (`agent_config`)** to store and reuse selected parameters  
- **Event-driven logic** triggered by a button click so the system runs only after the user finalizes inputs

### Why this matters
This layer makes the project function more like a **tool or service** rather than simple analysis code.  
It demonstrates that the system can start from **different datasets, population sizes, and policy scenarios**, supporting flexible simulations.

It lets the user choose the data source, synthetic population size, and policy scenario before the system runs.

# 2. Source Data Intake, Loading, and Initial Quality Audit

##### Loads the real source datasets and checks whether they are good enough to use.

In [ ]:
import pandas as pd
import numpy as np

# ____________________________________________________________
# Purpose:
# Load the three source datasets and run a first-pass
# audit to understand:
# - dataset size
# - missing values
# - duplicate rows
# - column types
# - overall structure
#
# This is the first real “agent service” step because the system
# is inspecting the source data before doing any cleaning or
# synthetic generation.
# ____________________________________________________________

# ____________________________________________________________
# Import required libraries
# - pandas = table/dataframe handling ***
# - numpy  = numerical support
# ____________________________________________________________
import pandas as pd
import numpy as np

# ____________________________________________________________
# Load source datasets from Colab storage
# Functions like read_csv(), .head(), .shape, ***
#and the audit checks help confirm the files loaded properly ***
# ____________________________________________________________
population_df = pd.read_csv("/content/population_demographics.csv")
healthcare_df = pd.read_csv("/content/healthcare_utilization.csv")
chronic_df = pd.read_csv("/content/chronic_disease_dataset.csv")

# ____________________________________________________________
# Confirm each dataset loaded correctly
# - Gives a quick shape check before deeper auditing (quality check) ***
# ____________________________________________________________
print("Population dataset shape:", population_df.shape)
print("Healthcare dataset shape:", healthcare_df.shape)
print("Chronic disease dataset shape:", chronic_df.shape)


Population dataset shape: (21, 6)
Healthcare dataset shape: (15757, 56)
Chronic disease dataset shape: (101766, 50)


In [ ]:
# ____________________________________________________________
# Preview the first few rows of dataset (confirm files open) ***
# - Helps visually confirm the columns and formatting
# ____________________________________________________________

population_df.head()


,age_group,2021,2022,2023,2024,2025
0,0 to 4 years,"1,902,883","1,887,192","1,874,658","1,874,025","1,871,184"
1,5 to 9 years,"2,072,565","2,090,692","2,120,248","2,151,780","2,138,350"
2,10 to 14 years,"2,114,016","2,146,637","2,187,386","2,232,438","2,251,628"
3,15 to 19 years,"2,059,975","2,140,739","2,230,085","2,313,964","2,319,393"
4,20 to 24 years,"2,404,398","2,465,156","2,589,895","2,741,716","2,703,780"


In [ ]:
# ____________________________________________________________
# Preview the first few rows of dataset
# - Helps visually confirm the columns and formatting
# ____________________________________________________________

healthcare_df.head()


,SNO,MRD No.,D.O.A,D.O.D,AGE,GENDER,RURAL,TYPE OF ADMISSION-EMERGENCY/OPD,month year,DURATION OF STAY,...,CONGENITAL,UTI,NEURO CARDIOGENIC SYNCOPE,ORTHOSTATIC,INFECTIVE ENDOCARDITIS,DVT,CARDIOGENIC SHOCK,SHOCK,PULMONARY EMBOLISM,CHEST INFECTION
0,1,234735,4/1/2017,4/3/2017,81,M,R,E,Apr-17,3,...,0,0,0,0,0,0,0,0,0,0
1,2,234696,4/1/2017,4/5/2017,65,M,R,E,Apr-17,5,...,0,0,0,0,0,0,0,0,0,0
2,3,234882,4/1/2017,4/3/2017,53,M,U,E,Apr-17,3,...,0,0,0,0,0,0,0,0,0,0
3,4,234635,4/1/2017,4/8/2017,67,F,U,E,Apr-17,8,...,0,0,0,0,0,0,0,0,0,0
4,5,234486,4/1/2017,4/23/2017,60,F,U,E,Apr-17,23,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ____________________________________________________________
# Preview the first few rows of dataset
# - Helps visually confirm the columns and formatting
# ____________________________________________________________

chronic_df.head()


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [ ]:
# ____________________________________________________________
# Define a reusable audit function
# - This function gives a quick quality summary for any dataset
# ____________________________________________________________
def audit_dataset(df, name):
    print("=" * 30)
    print("DATASET AUDIT:", name)
    print("=" * 30)

    # Basic shape
    print("\nShape:")
    print(df.shape)

    # Missing value count by column
    print("\nMissing values:")
    print(df.isnull().sum())

    # Duplicate row count
    print("\nDuplicate rows:")
    print(df.duplicated().sum())

    # Data types
    print("\nColumn types:")
    print(df.dtypes)

    # General summary stats
    print("\nBasic statistics:")
    print(df.describe(include="all"))


In [ ]:
# ____________________________________________________________
# Run the audit on source dataset ***
# - This is the first-pass agent inspection stage
# - Some year columns are stored as text instead of numbers ***
# ____________________________________________________________

audit_dataset(population_df, "Population Demographics")


DATASET AUDIT: Population Demographics

Shape:
(21, 6)

Missing values:
age_group    0
2021         0
2022         0
2023         0
2024         0
2025         0
dtype: int64

Duplicate rows:
0

Column types:
age_group    object
2021         object
2022         object
2023         object
2024         object
2025         object
dtype: object

Basic statistics:
           age_group       2021       2022       2023       2024       2025
count             21         21         21         21         21         21
unique            21         21         21         21         21         21
top     0 to 4 years  1,902,883  1,887,192  1,874,658  1,874,025  1,871,184
freq               1          1          1          1          1          1


In [ ]:
# ____________________________________________________________
# Run the audit on source dataset
# - This is the first-pass agent inspection stage
# - Missing clinical values ***
# ____________________________________________________________

audit_dataset(healthcare_df, "Healthcare Utilization")


DATASET AUDIT: Healthcare Utilization

Shape:
(15757, 56)

Missing values:
SNO                                   0
MRD No.                               0
D.O.A                                 0
D.O.D                                 0
AGE                                   0
GENDER                                0
RURAL                                 0
TYPE OF ADMISSION-EMERGENCY/OPD       0
month year                            0
DURATION OF STAY                      0
duration of intensive unit stay       0
OUTCOME                               0
SMOKING                               0
ALCOHOL                               0
DM                                    0
HTN                                   0
CAD                                   0
PRIOR CMP                             0
CKD                                   0
HB                                  252
TLC                                 286
PLATELETS                           285
GLUCOSE                             863
UREA 

In [ ]:
# ____________________________________________________________
# Run the audit on source dataset
# - This is the first-pass agent inspection stage
# - Lot of missing data ***
# ____________________________________________________________

audit_dataset(chronic_df, "Chronic Disease Dataset")


DATASET AUDIT: Chronic Disease Dataset

Shape:
(101766, 50)

Missing values:
encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlor

### What this section does
This step **loads the three source datasets into Python** and previews them to confirm that the files were imported correctly.  
It then runs a reusable **`audit_dataset()` function** on each dataset to examine the structure of the data before any cleaning or synthetic generation occurs.
*   *Shows what needs cleaning*

### Why this step exists
The system must first ***evaluate the quality of the incoming data**** before it can safely generate synthetic healthcare populations.  
By auditing the datasets at the beginning of the pipeline, the service ensures that it does **not blindly trust incoming data** and can detect potential hygiene or structural issues early.

### Technologies used
- **`pandas` DataFrame operations** to load and analyze tabular CSV datasets  
- A reusable **`audit_dataset()` function** so the same audit logic can be applied consistently across multiple datasets  
- **Exploratory quality-check analysis**, which is standard practice in real-world data pipelines before preprocessing

### Why this matters
This step marks the point where the project begins functioning like an **agentic data service rather than simple analysis code**.  
The system first **inspects and validates the input data**, which justifies the later stages of cleaning, governance checks, and synthetic population generation.

The system loads the datasets and audits their structure to ensure the source data is reliable before any synthetic generation begins.

*  *Shows governance-aware*
*  *quality control stage of pipeline*



# 3. Data Cleaning and Governance

##### This step cleans the raw datasets and turns them into safe, usable inputs for the rest of the pipeline.

In [ ]:
# ____________________________________________________________
# Purpose:
# Take the raw source datasets from Step 3 and prepare them for
# safe synthetic generation by:
# - creating protected working copies
# - cleaning numeric/text formatting issues
# - removing irrelevant or sensitive columns
# - standardizing column names
# - converting placeholder values into true missing values
# - checking for potential identifier fields
#
# Clean messy data - prevent incorrect synthetic patients***
# This step completes the “data hygiene + governance” requirement
# of the hackathon and ensures the datasets are suitable for
# synthetic population generation.
# ____________________________________________________________

# ____________________________________________________________
# Create working copies of the original datasets
# - Protects the raw data and lets us clean safely
# ____________________________________________________________
population_clean = population_df.copy()
healthcare_clean = healthcare_df.copy()
chronic_clean = chronic_df.copy()


In [ ]:
# ____________________________________________________________
# Clean the population dataset (using pandas data cleaning)***
# - Fix column names and convert year columns from text to numbers
# ____________________________________________________________

# Remove leading/trailing spaces from column names
population_clean.columns = [str(col).strip() for col in population_clean.columns]

# Define the year columns we want to clean
year_cols = ["2021", "2022", "2023", "2024", "2025"]

# Convert each year column from text (with commas) to numeric
for col in year_cols:
    population_clean[col] = (
        population_clean[col]

        # make sure values are strings first
        .astype(str)

        # remove weird commas
        .str.replace(",", "", regex=False)

        # remove extra spaces
        .str.strip()
    )

    population_clean[col] = pd.to_numeric(population_clean[col], errors="coerce")

# Check results
print(population_clean.dtypes)
population_clean.head()

# The demographics dataset is cleaned and simplified into a usable format***

age_group    object
2021          int64
2022          int64
2023          int64
2024          int64
2025          int64
dtype: object


,age_group,2021,2022,2023,2024,2025
0,0 to 4 years,1902883,1887192,1874658,1874025,1871184
1,5 to 9 years,2072565,2090692,2120248,2151780,2138350
2,10 to 14 years,2114016,2146637,2187386,2232438,2251628
3,15 to 19 years,2059975,2140739,2230085,2313964,2319393
4,20 to 24 years,2404398,2465156,2589895,2741716,2703780


In [ ]:
# ____________________________________________________________
# Simplify the population dataset for synthetic generation
# - Keep only age group and one population column (2024)
# ____________________________________________________________
population_clean["population"] = population_clean["2024"]
population_clean = population_clean[["age_group", "population"]]

population_clean.head()


,age_group,population
0,0 to 4 years,1874025
1,5 to 9 years,2151780
2,10 to 14 years,2232438
3,15 to 19 years,2313964
4,20 to 24 years,2741716


In [ ]:
# ____________________________________________________________
# Remove identifier-like or irrelevant healthcare columns
# - These fields are not useful for modeling and may create
#   unnecessary governance concerns.
# ____________________________________________________________
cols_to_drop_healthcare = [
    "SNO",
    "MRD No.",
    "D.O.A",
    "D.O.D",
    "month year"
]

healthcare_clean = healthcare_clean.drop(
    columns=[col for col in cols_to_drop_healthcare if col in healthcare_clean.columns],
    errors="ignore"
)


In [ ]:
# ____________________________________________________________
# Standardize healthcare column names
# - Make them lowercase and easier to reference later
# ____________________________________________________________
healthcare_clean.columns = (
    healthcare_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print(healthcare_clean.columns.tolist())

# Column names are cleaned into a consistent, machine-friendly format***

['age', 'gender', 'rural', 'type_of_admission_emergency_opd', 'duration_of_stay', 'duration_of_intensive_unit_stay', 'outcome', 'smoking', 'alcohol', 'dm', 'htn', 'cad', 'prior_cmp', 'ckd', 'hb', 'tlc', 'platelets', 'glucose', 'urea', 'creatinine', 'bnp', 'raised_cardiac_enzymes', 'ef', 'severe_anaemia', 'anaemia', 'stable_angina', 'acs', 'stemi', 'atypical_chest_pain', 'heart_failure', 'hfref', 'hfnef', 'valvular', 'chb', 'sss', 'aki', 'cva_infract', 'cva_bleed', 'af', 'vt', 'psvt', 'congenital', 'uti', 'neuro_cardiogenic_syncope', 'orthostatic', 'infective_endocarditis', 'dvt', 'cardiogenic_shock', 'shock', 'pulmonary_embolism', 'chest_infection']


In [ ]:
# ____________________________________________________________
# Convert numeric-looking text columns into numeric values
# - Some columns may look like text but really contain numbers
# ____________________________________________________________
for col in healthcare_clean.columns:
    if healthcare_clean[col].dtype == "object":
        healthcare_clean[col] = healthcare_clean[col].astype(str).str.strip()

        # Remove commas if present
        cleaned_series = healthcare_clean[col].str.replace(",", "", regex=False)

        # Attempt numeric conversion
        numeric_version = pd.to_numeric(cleaned_series, errors="coerce")

        # Only convert if most values are numeric-like
        if numeric_version.notna().mean() > 0.7:
            healthcare_clean[col] = numeric_version

# Recheck missingness
healthcare_clean.isnull().sum().sort_values(ascending=False).head(20)

# The system identifies which variables have missing data and severity***

,0
ef,1599
glucose,945
platelets,294
tlc,290
hb,256
creatinine,251
urea,244
chest_infection,1
rural,0
age,0


In [ ]:
# ____________________________________________________________
# Standardize chronic disease dataset column names
# ____________________________________________________________
chronic_clean.columns = (
    chronic_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)


In [ ]:
# ____________________________________________________________
# Convert placeholder values into true missing values
# - The chronic dataset uses symbols like "?" and text placeholders
# ____________________________________________________________
chronic_clean = chronic_clean.replace("?", np.nan)
chronic_clean = chronic_clean.replace("None", np.nan)
chronic_clean = chronic_clean.replace("Unknown/Invalid", np.nan)

# Check missing values after cleaning placeholders
chronic_clean.isnull().sum().sort_values(ascending=False).head(20)


,0
weight,98569
max_glu_serum,96420
a1cresult,84748
medical_specialty,49949
payer_code,40256
race,2273
diag_3,1423
diag_2,358
diag_1,21
gender,3


In [ ]:
# ____________________________________________________________
# Identify columns with extremely high missingness
# - We will drop columns with more than 40% missing values***
# ____________________________________________________________
missing_pct = chronic_clean.isnull().mean() * 100

high_missing_cols = missing_pct[missing_pct > 40].index.tolist()

print("Columns with >40% missing values:")
print(high_missing_cols)


Columns with >40% missing values:
['weight', 'medical_specialty', 'max_glu_serum', 'a1cresult']


In [ ]:
# ____________________________________________________________
# Drop very high-missingness columns
# ____________________________________________________________

chronic_clean = chronic_clean.drop(columns=high_missing_cols, errors="ignore")


In [ ]:
# ____________________________________________________________
# Drop obvious identifier columns (reduce noice)***
# ____________________________________________________________
chronic_clean = chronic_clean.drop(
    columns=[col for col in ["encounter_id", "patient_nbr"] if col in chronic_clean.columns],
    errors="ignore"
)

print(chronic_clean.shape)
chronic_clean.head()


(101766, 44)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,...,citoglipton,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,diabetesmed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,NaN,41,0,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,NaN,59,0,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,11,5,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,NaN,44,1,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,NaN,51,0,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [ ]:
# ____________________________________________________________
# Create a governance screening function
# - Checks whether any column names look like identifiers
# ____________________________________________________________
def governance_check(df, name):
    risky_keywords = [
        "name", "address", "phone", "email",
        "sin", "social", "mrn", "dob",
        "date_of_birth", "patient_nbr",
        "encounter_id", "mrd"
    ]

    found_risky_cols = []

    for col in df.columns:
        col_lower = col.lower()
        for keyword in risky_keywords:
            if keyword in col_lower:
                found_risky_cols.append(col)
                break

    print("=================================")
    print("GOVERNANCE CHECK:", name)
    print("=================================")

    if found_risky_cols:
        print("Potential sensitive columns found:")
        print(found_risky_cols)
        print("Status: REVIEW NEEDED")
    else:
        print("No direct identifiers detected.")
        print("Status: APPROVED FOR SYNTHETIC GENERATION")


In [ ]:
# ____________________________________________________________
# Run governance checks on all cleaned datasets***
# ____________________________________________________________
governance_check(population_clean, "Population Demographics")
governance_check(healthcare_clean, "Healthcare Utilization")
governance_check(chronic_clean, "Chronic Disease Dataset")


GOVERNANCE CHECK: Population Demographics
No direct identifiers detected.
Status: APPROVED FOR SYNTHETIC GENERATION
GOVERNANCE CHECK: Healthcare Utilization
No direct identifiers detected.
Status: APPROVED FOR SYNTHETIC GENERATION
GOVERNANCE CHECK: Chronic Disease Dataset
No direct identifiers detected.
Status: APPROVED FOR SYNTHETIC GENERATION


In [ ]:
# ____________________________________________________________
# Re-audit cleaned dataset
# - Verifies that cleaning improved the data structure (improved)***
# ____________________________________________________________

audit_dataset(population_clean, "Population Demographics - Cleaned")


DATASET AUDIT: Population Demographics - Cleaned

Shape:
(21, 2)

Missing values:
age_group     0
population    0
dtype: int64

Duplicate rows:
0

Column types:
age_group     object
population     int64
dtype: object

Basic statistics:
           age_group    population
count             21  2.100000e+01
unique            21           NaN
top     0 to 4 years           NaN
freq               1           NaN
mean             NaN  1.964873e+06
std              NaN  1.001553e+06
min              NaN  1.191900e+04
25%              NaN  1.535283e+06
50%              NaN  2.313964e+06
75%              NaN  2.712778e+06
max              NaN  3.145235e+06


In [ ]:
# ____________________________________________________________
# Re-audit cleaned dataset
# - Verifies that cleaning improved the data structure
# ____________________________________________________________

audit_dataset(healthcare_clean, "Healthcare Utilization - Cleaned")


DATASET AUDIT: Healthcare Utilization - Cleaned

Shape:
(15757, 51)

Missing values:
age                                   0
gender                                0
rural                                 0
type_of_admission_emergency_opd       0
duration_of_stay                      0
duration_of_intensive_unit_stay       0
outcome                               0
smoking                               0
alcohol                               0
dm                                    0
htn                                   0
cad                                   0
prior_cmp                             0
ckd                                   0
hb                                  256
tlc                                 290
platelets                           294
glucose                             945
urea                                244
creatinine                          251
bnp                                   0
raised_cardiac_enzymes                0
ef                                 

In [ ]:
# ____________________________________________________________
# Re-audit cleaned dataset
# - Verifies that cleaning improved the data structure
# ____________________________________________________________

audit_dataset(chronic_clean, "Chronic Disease Dataset - Cleaned")


DATASET AUDIT: Chronic Disease Dataset - Cleaned

Shape:
(101766, 44)

Missing values:
race                         2273
gender                          3
age                             0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide                   0
glipizide                       0
glyburide                       0
tolbutamide                  

In [ ]:
# ____________________________________________________________
# 15. Save cleaned datasets for later pipeline stages
# ____________________________________________________________
population_clean.to_csv("population_clean.csv", index=False)
healthcare_clean.to_csv("healthcare_clean.csv", index=False)
chronic_clean.to_csv("chronic_clean.csv", index=False)


### What this section does
This step transforms the **raw source data into clean, governance-reviewed, analysis-ready datasets**.  
It performs four key tasks:

- **Fixes formatting inconsistencies**
- **Handles missing values**
- **Removes risky or irrelevant fields**
- **Applies governance checks before synthetic generation**
- ***Clean safe and reliable data.***

### Why this step exists
The project is designed to function as a **responsible synthetic healthcare data service**, not just a data generator.  
Before generating synthetic populations, the system must ensure that the source data is ***clean, structured, and trustworthy***.

This step directly fulfills the requirement that the system:

- Identify **data hygiene issues**
- Perform **governance screening**
- Ensure the **cleaned data reflects approved metadata**

### Technologies used
This stage uses common **data pipeline preprocessing techniques**, including:

- **Data cleaning and preprocessing**
- **String normalization**
- **Missing-value handling**
- **Rule-based governance screening**
- **Reusable function-based auditing**

These are standard practices in **real-world data engineering pipelines before modeling or simulation**.

### Why this matters
This is one of the most important sections of the notebook because it makes the project:

- ***Ethical***
- ***Credible***
- ***Policy-aware***
- ***Appropriate for healthcare data (remove messy data)***

Without this step, the workflow would simply look like:
> “We took some data and generated synthetic patients.”

With this step, the workflow becomes:
> “We built an agentic healthcare data service that audits, cleans, approves, and transforms source data before synthetic generation.”


**This block is the data preparation and governance stage.**  
The agent cleans the datasets, standardizes formats, handles missing values, removes risky identifiers, and ensures the data is safe to use for synthetic population generation.

# 4. Synthetic Population Generation

##### Creates a fully synthetic patient population from real-world data patterns which are realistic, privacy-safe synthetic patients with demographics, health conditions, and access barriers.

In [ ]:
# ____________________________________________________________
# Step 4 — Synthetic Population Generation Engine
# ____________________________________________________________
# Purpose:
# Generate a synthetic Southlake-like patient population using
# cleaned public data patterns rather than real patient records.
#
# This step creates fake but realistic patients by:
# - using the real age-group distribution from population data
# - sampling demographic features
# - assigning chronic disease flags probabilistically
# - adding access and equity variables
#
# This is the core synthetic data creation layer of the project.
# Because patient-level data needed to simulate healthcare systems, ***
# but you cannot safely use real patient records
# ____________________________________________________________
import pandas as pd
import numpy as np


In [ ]:
# ____________________________________________________________
# Prepare the age distribution from cleaned population data
# - Synthetic age structure should mirror the real source
#   population rather than being randomly uniform
# Uses probability distributions and rule-based logic to generate patients***
# System converts real population data into probabilities,
# Samples patients using NumPy, and then assigns features using patterns
# ____________________________________________________________

# Make a clean copy of the population dataset
age_dist = population_clean.copy()

# Remove any rows missing age group or population
age_dist = age_dist.dropna(subset=["age_group", "population"])

# Convert the population counts into probabilities, everything after uses this*
age_dist["probability"] = age_dist["population"] / age_dist["population"].sum()

# Preview the age sampling table
age_dist

# The system converts real population counts into probabilities for sampling***
# ensures more common age groups appear more often in the synthetic population r

,age_group,population,probability
0,0 to 4 years,1874025,0.045417
1,5 to 9 years,2151780,0.052149
2,10 to 14 years,2232438,0.054104
3,15 to 19 years,2313964,0.056079
4,20 to 24 years,2741716,0.066446
5,25 to 29 years,3031487,0.073469
6,30 to 34 years,3145235,0.076225
7,35 to 39 years,2950432,0.071504
8,40 to 44 years,2796675,0.067778
9,45 to 49 years,2538433,0.061519


In [ ]:
# ____________________________________________________________
# Set the number of synthetic patients to generate
# - If control panel was used, pull from agent_config
# - Otherwise fall back to a default value.
# ____________________________________________________________

if "agent_config" in globals() and "population_size" in agent_config:
    n_patients = agent_config["population_size"]
else:
    print("Control panel not used — defaulting to 10,000 synthetic patients")
    n_patients = 10000

print("Synthetic population size:", n_patients)

# The system sets and confirms how many synthetic patients to generate***

Control panel not used — defaulting to 10,000 synthetic patients
Synthetic population size: 10000


In [ ]:
# ____________________________________________________________
# Create the base synthetic patient table
# This generates:
# - patient_id
# - age_group
# - sex
# - rurality
#
# - Age is sampled using the real population distribution
# - Sex and rurality are sampled using simple population-level
#   assumptions.
# ____________________________________________________________
np.random.seed(42)  # for reproducibility

synthetic_patients = pd.DataFrame({
    "patient_id": [f"P{i+1:05d}" for i in range(n_patients)],

    "age_group": np.random.choice(
        age_dist["age_group"],
        size=n_patients,
        p=age_dist["probability"]
    ),

    "sex": np.random.choice(
        ["Female", "Male"],
        size=n_patients,
        p=[0.51, 0.49]
    ),

    "rurality": np.random.choice(
        ["Urban", "Rural"],
        size=n_patients,
        p=[0.82, 0.18]
    )
})

synthetic_patients.head()

# First synthetic patients are generated with basic demographic attributes***

,patient_id,age_group,sex,rurality
0,P00001,30 to 34 years,Female,Urban
1,P00002,75 to 79 years,Female,Urban
2,P00003,55 to 59 years,Female,Urban
3,P00004,45 to 49 years,Male,Urban
4,P00005,15 to 19 years,Female,Urban


In [ ]:
# ____________________________________________________________
# Map each age group to a simple age-related risk score
# - Older age groups receive higher values because they are more
#   likely to face chronic conditions, utilization, and complexity.
# ____________________________________________________________
age_risk_map = {
    "0 to 4 years": 0.05,
    "5 to 9 years": 0.05,
    "10 to 14 years": 0.05,
    "15 to 19 years": 0.08,
    "20 to 24 years": 0.10,
    "25 to 29 years": 0.12,
    "30 to 34 years": 0.15,
    "35 to 39 years": 0.18,
    "40 to 44 years": 0.22,
    "45 to 49 years": 0.28,
    "50 to 54 years": 0.35,
    "55 to 59 years": 0.42,
    "60 to 64 years": 0.50,
    "65 to 69 years": 0.58,
    "70 to 74 years": 0.66,
    "75 to 79 years": 0.74,
    "80 to 84 years": 0.82,
    "85 to 89 years": 0.88,
    "90 to 94 years": 0.92,
    "95 to 99 years": 0.95,
    "100 years and over": 0.97
}

synthetic_patients["age_risk_score"] = synthetic_patients["age_group"].map(age_risk_map)


In [ ]:
# ____________________________________________________________
# Generate synthetic chronic disease and access variables
# - Disease flags are probabilistically linked to age risk
# - Access variables reflect realistic equity barriers
# ____________________________________________________________

# Diabetes probability increases with age
synthetic_patients["diabetes_flag"] = np.where(
    np.random.rand(n_patients) < (0.08 + 0.35 * synthetic_patients["age_risk_score"]),
    1, 0
)

# Hypertension probability increases with age
synthetic_patients["hypertension_flag"] = np.where(
    np.random.rand(n_patients) < (0.10 + 0.45 * synthetic_patients["age_risk_score"]),
    1, 0
)

# COPD probability increases with age, but remains lower overall
synthetic_patients["copd_flag"] = np.where(
    np.random.rand(n_patients) < (0.03 + 0.20 * synthetic_patients["age_risk_score"]),
    1, 0
)

# Heart disease probability increases with age
synthetic_patients["heart_disease_flag"] = np.where(
    np.random.rand(n_patients) < (0.04 + 0.25 * synthetic_patients["age_risk_score"]),
    1, 0
)

# Mental health prevalence is modeled more broadly
synthetic_patients["mental_health_flag"] = np.where(
    np.random.rand(n_patients) < 0.12,
    1, 0
)

# ____________________________________________________________
# Digital access:
# Rural + older patients are more likely to have weaker digital access
# ____________________________________________________________
synthetic_patients["digital_access"] = np.where(
    (synthetic_patients["rurality"] == "Rural") & (synthetic_patients["age_risk_score"] > 0.6),
    np.random.choice(["Low", "Medium"], size=n_patients),
    np.random.choice(["Low", "Medium", "High"], size=n_patients, p=[0.15, 0.35, 0.50])
)

# ____________________________________________________________
# Primary care access:
# Rural patients are more likely to have limited access
# ____________________________________________________________
synthetic_patients["primary_care_access"] = np.where(
    synthetic_patients["rurality"] == "Rural",
    np.random.choice(["Good", "Limited"], size=n_patients, p=[0.45, 0.55]),
    np.random.choice(["Good", "Limited"], size=n_patients, p=[0.70, 0.30])
)


In [ ]:
# ____________________________________________________________
# Count how many chronic conditions each patient has
# - This gives a simple measure of multi-morbidity
# ____________________________________________________________
condition_cols = [
    "diabetes_flag",
    "hypertension_flag",
    "copd_flag",
    "heart_disease_flag",
    "mental_health_flag"
]

synthetic_patients["chronic_condition_count"] = synthetic_patients[condition_cols].sum(axis=1)


In [ ]:
# ____________________________________________________________
# Add two important access barrier variables:
# - mobility_limitation
# - transport_barrier
#
# These are critical for routing decisions later.
# ____________________________________________________________

# Mobility limitations increase with age
synthetic_patients["mobility_limitation"] = np.where(
    np.random.rand(n_patients) < (0.03 + 0.40 * synthetic_patients["age_risk_score"]),
    1, 0
)

# Transportation barriers are more likely in rural areas
synthetic_patients["transport_barrier"] = np.where(
    synthetic_patients["rurality"] == "Rural",
    np.random.choice([0, 1], size=n_patients, p=[0.45, 0.55]),
    np.random.choice([0, 1], size=n_patients, p=[0.80, 0.20])
)


In [ ]:
# ____________________________________________________________
# Review the synthetic dataset
# - Confirm shape and preview first rows
# ____________________________________________________________
print(synthetic_patients.shape)
synthetic_patients.head(10)

# The synthetic population is enriched with health, risk, and access features***

(10000, 15)


,patient_id,age_group,sex,rurality,age_risk_score,diabetes_flag,hypertension_flag,copd_flag,heart_disease_flag,mental_health_flag,digital_access,primary_care_access,chronic_condition_count,mobility_limitation,transport_barrier
0,P00001,30 to 34 years,Female,Urban,0.15,0,0,0,0,1,Low,Limited,1,0,0
1,P00002,75 to 79 years,Female,Urban,0.74,0,1,0,0,0,High,Limited,1,0,0
2,P00003,55 to 59 years,Female,Urban,0.42,0,1,0,0,0,High,Good,1,1,0
3,P00004,45 to 49 years,Male,Urban,0.28,0,1,0,0,0,High,Good,1,0,0
4,P00005,15 to 19 years,Female,Urban,0.08,0,0,0,0,0,High,Good,0,1,0
5,P00006,15 to 19 years,Male,Urban,0.08,0,0,0,0,0,High,Good,0,0,0
6,P00007,5 to 9 years,Female,Rural,0.05,0,0,0,0,0,Medium,Good,0,1,1
7,P00008,65 to 69 years,Male,Urban,0.58,0,0,0,0,0,Medium,Good,0,0,0
8,P00009,45 to 49 years,Male,Urban,0.28,0,0,0,0,0,Low,Limited,0,0,0
9,P00010,55 to 59 years,Male,Urban,0.42,0,0,0,0,0,High,Good,0,0,0


In [ ]:
# ____________________________________________________________
# Check the distributions of key synthetic variables
# - Helps validate realism.
# ____________________________________________________________
print(synthetic_patients["age_group"].value_counts(normalize=True).sort_index())
print(synthetic_patients["rurality"].value_counts(normalize=True))
print(synthetic_patients["diabetes_flag"].value_counts(normalize=True))
print(synthetic_patients["chronic_condition_count"].value_counts(normalize=True).sort_index())

# The system checks whether the synthetic population looks realistic***

age_group
0 to 4 years           0.0454
10 to 14 years         0.0540
100 years and older    0.0001
15 to 19 years         0.0572
20 to 24 years         0.0680
25 to 29 years         0.0733
30 to 34 years         0.0784
35 to 39 years         0.0713
40 to 44 years         0.0699
45 to 49 years         0.0622
5 to 9 years           0.0554
50 to 54 years         0.0597
55 to 59 years         0.0592
60 to 64 years         0.0621
65 to 69 years         0.0577
70 to 74 years         0.0456
75 to 79 years         0.0376
80 to 84 years         0.0235
85 to 89 years         0.0118
90 to 94 years         0.0061
95 to 99 years         0.0015
Name: proportion, dtype: float64
rurality
Urban    0.8238
Rural    0.1762
Name: proportion, dtype: float64
diabetes_flag
0    0.8176
1    0.1824
Name: proportion, dtype: float64
chronic_condition_count
0    0.4644
1    0.3715
2    0.1313
3    0.0285
4    0.0042
5    0.0001
Name: proportion, dtype: float64


In [ ]:
# ____________________________________________________________
# Save the synthetic patient population
# - Creates a reusable output file for later steps
# ____________________________________________________________

synthetic_patients.to_csv("synthetic_patients_step4.csv", index=False)


### What this section does
This step is the **synthetic data creation engine** of the system.

*Synthetic populations let us simulate real-world patients and healthcare demand without using any real personal data, making the system safe and privacy-compliant.*

Using the cleaned source data, the agent generates a **fully artificial patient population** that reflects realistic population patterns.  
The synthetic population includes features such as:

- **Demographics**
- **Chronic health conditions**
- **Access-to-care barriers**
- **Mobility limitations**
- **Rural vs. urban location**
- **Healthcare access characteristics**

At this stage, the notebook transitions from **data preparation to synthetic population generation**.

### Why this step exists
The project is designed to support **healthcare innovation without using real patient data**.  
To do this, the system must be able to **generate realistic but entirely synthetic healthcare populations on demand**.

This step fulfills the hackathon requirement of creating **privacy-safe synthetic healthcare data** that can be used for simulation and planning.

### Technologies used
This stage relies on transparent and explainable synthetic data methods, including:

- **Weighted random sampling** from real-world public distributions  
- **Probabilistic rule-based synthetic generation**
- **Feature engineering** to construct realistic patient attributes
- **Distribution validation** to ensure the synthetic population reflects plausible patterns

Rather than using deep generative models (such as GANs), the system uses **distribution-based sampling with rule assignment**, which provides:

- **Explainability**
- **Fast generation**
- **Practical implementation**
- **Ethical transparency**

*It was generated by converting real population data into probabilities and using those to randomly create patients, then adding health conditions and access factors using rule-based logic.*

### Why this matters
This step is where **real population patterns are transformed into synthetic healthcare entities**.

Before this stage, the notebook only contains **cleaned source datasets**.  
After this stage, the system produces a **new synthetic Southlake-like patient population** that can be used for downstream simulation and healthcare policy testing.


**This block is the synthetic population generator.**  
The agent uses cleaned public data patterns to create a realistic, fully artificial patient population with demographic, chronic disease, and healthcare access attributes.

# 5. Predictive Risk Analysis and Patient Risk Scoring

###### This step turns the synthetic population into a risk-aware population. The system calculates multiple healthcare risk scores for every synthetic patient. Healthcare decisions depend on understanding risk, not just patient characteristics.
##### *It evaluates factors like age, chronic conditions, and access barriers to estimate how likely each patient is to need care or face challenges.*

In [ ]:
# ____________________________________________________________
# Purpose:
# Take the synthetic patient population from 'Synthetic Population Generation'
# and assign planning-level risk scores that help estimate:
# - ED escalation risk
# - readmission vulnerability
# - avoidable utilization risk
# - care complexity
#
# These are not clinical diagnoses or real patient predictions.
# They are synthetic, explainable planning scores used to power
# routing and system simulation.
# ____________________________________________________________

# ____________________________________________________________
# Create a working copy of the synthetic patient table
# - This protects the 'Synthetic Population Generation' output
#   from accidental overwriting.
# ____________________________________________________________

risk_df = synthetic_patients.copy()


In [ ]:
# ____________________________________________________________
# Map access variables into numeric scores
# - These scores let us use access barriers directly in formulas
# - Higher scores = greater access-related risk.
# ____________________________________________________________

digital_access_score_map = {
    "Low": 1.0,
    "Medium": 0.5,
    "High": 0.0
}

primary_care_access_score_map = {
    "Limited": 1.0,
    "Good": 0.0
}

risk_df["digital_access_score"] = risk_df["digital_access"].map(digital_access_score_map)
risk_df["primary_care_access_score"] = risk_df["primary_care_access"].map(primary_care_access_score_map)


In [ ]:
# ____________________________________________________________
# Create a planning-level Emergency Department escalation risk
# - Estimates who is more likely to need ED-level care
# It transforms raw patient data into actionable insights for healthcare***
# planning. allows the system to predict demand, guide care routing,
# and support policy decisions.
# ____________________________________________________________
risk_df["ed_escalation_risk_score"] = (
    0.25 * risk_df["age_risk_score"] +
    0.15 * risk_df["diabetes_flag"] +
    0.10 * risk_df["hypertension_flag"] +
    0.15 * risk_df["copd_flag"] +
    0.15 * risk_df["heart_disease_flag"] +
    0.10 * risk_df["mental_health_flag"] +
    0.15 * (risk_df["chronic_condition_count"] / 5) +
    0.10 * risk_df["mobility_limitation"] +
    0.10 * risk_df["transport_barrier"] +
    0.10 * risk_df["primary_care_access_score"]
)

# Keep the score between 0 and 1
risk_df["ed_escalation_risk_score"] = risk_df["ed_escalation_risk_score"].clip(0, 1)


In [ ]:
# ____________________________________________________________
# Create readmission vulnerability score
# - Estimates who may struggle more after care transitions
# ____________________________________________________________
risk_df["readmission_vulnerability_score"] = (
    0.20 * risk_df["age_risk_score"] +
    0.20 * (risk_df["chronic_condition_count"] / 5) +
    0.15 * risk_df["diabetes_flag"] +
    0.10 * risk_df["heart_disease_flag"] +
    0.10 * risk_df["copd_flag"] +
    0.10 * risk_df["mobility_limitation"] +
    0.10 * risk_df["primary_care_access_score"] +
    0.10 * risk_df["digital_access_score"]
)

risk_df["readmission_vulnerability_score"] = risk_df["readmission_vulnerability_score"].clip(0, 1)


In [ ]:
# ____________________________________________________________
# Create avoidable utilization risk score
# - Estimates who may use hospital-based care for issues that
#   could potentially be managed earlier or elsewhere.
# ____________________________________________________________
risk_df["avoidable_utilization_risk_score"] = (
    0.15 * risk_df["age_risk_score"] +
    0.10 * risk_df["diabetes_flag"] +
    0.10 * risk_df["hypertension_flag"] +
    0.10 * risk_df["mental_health_flag"] +
    0.20 * risk_df["primary_care_access_score"] +
    0.20 * risk_df["digital_access_score"] +
    0.15 * risk_df["transport_barrier"] +
    0.10 * risk_df["rurality"].map({"Urban": 0, "Rural": 1})
)

risk_df["avoidable_utilization_risk_score"] = risk_df["avoidable_utilization_risk_score"].clip(0, 1)


In [ ]:
# ____________________________________________________________
# Create overall care complexity score
# - This reflects how difficult a patient may be to manage across
#   care settings.
# ____________________________________________________________
risk_df["care_complexity_score"] = (
    0.20 * risk_df["age_risk_score"] +
    0.25 * (risk_df["chronic_condition_count"] / 5) +
    0.10 * risk_df["mobility_limitation"] +
    0.10 * risk_df["transport_barrier"] +
    0.10 * risk_df["primary_care_access_score"] +
    0.10 * risk_df["digital_access_score"] +
    0.15 * (
        risk_df["heart_disease_flag"] +
        risk_df["copd_flag"] +
        risk_df["mental_health_flag"]
    ) / 3
)

risk_df["care_complexity_score"] = risk_df["care_complexity_score"].clip(0, 1)


In [ ]:
# ____________________________________________________________
# Convert numeric scores into simple risk bands
# - Makes outputs easier to interpret and present.
# ____________________________________________________________
def risk_band(score):
    if score < 0.33:
        return "Low"
    elif score < 0.66:
        return "Medium"
    else:
        return "High"

risk_df["ed_escalation_risk_level"] = risk_df["ed_escalation_risk_score"].apply(risk_band)
risk_df["readmission_vulnerability_level"] = risk_df["readmission_vulnerability_score"].apply(risk_band)
risk_df["avoidable_utilization_risk_level"] = risk_df["avoidable_utilization_risk_score"].apply(risk_band)
risk_df["care_complexity_level"] = risk_df["care_complexity_score"].apply(risk_band)


In [ ]:
# ____________________________________________________________
# Identify the top risk driver for each patient
# - Makes the risk engine more transparent and explainable.
# ____________________________________________________________
def get_top_driver(row):
    driver_scores = {
        "Age": row["age_risk_score"],
        "Diabetes": row["diabetes_flag"] * 0.8,
        "Hypertension": row["hypertension_flag"] * 0.6,
        "COPD": row["copd_flag"] * 0.9,
        "Heart Disease": row["heart_disease_flag"] * 0.9,
        "Mental Health": row["mental_health_flag"] * 0.7,
        "Multi-morbidity": row["chronic_condition_count"] / 5,
        "Low Digital Access": row["digital_access_score"],
        "Limited Primary Care Access": row["primary_care_access_score"],
        "Mobility Limitation": row["mobility_limitation"] * 0.8,
        "Transport Barrier": row["transport_barrier"] * 0.8
    }

    return max(driver_scores, key=driver_scores.get)

risk_df["top_risk_driver"] = risk_df.apply(get_top_driver, axis=1)


In [ ]:
# ____________________________________________________________
# Inspect a sample of the risk-scored patients
# ____________________________________________________________
risk_df[[
    "patient_id",
    "age_group",
    "rurality",
    "chronic_condition_count",
    "ed_escalation_risk_score",
    "ed_escalation_risk_level",
    "readmission_vulnerability_score",
    "readmission_vulnerability_level",
    "avoidable_utilization_risk_score",
    "avoidable_utilization_risk_level",
    "care_complexity_score",
    "care_complexity_level",
    "top_risk_driver"
]].head(10)

# Each synthetic patient now has multiple risk scores and a top risk driver***

,patient_id,age_group,rurality,chronic_condition_count,ed_escalation_risk_score,ed_escalation_risk_level,readmission_vulnerability_score,readmission_vulnerability_level,avoidable_utilization_risk_score,avoidable_utilization_risk_level,care_complexity_score,care_complexity_level,top_risk_driver
0,P00001,30 to 34 years,Urban,1,0.2675,Low,0.270,Low,0.5225,Medium,0.330,Medium,Low Digital Access
1,P00002,75 to 79 years,Urban,1,0.4150,Medium,0.288,Low,0.4110,Medium,0.298,Low,Limited Primary Care Access
2,P00003,55 to 59 years,Urban,1,0.3350,Medium,0.224,Low,0.1630,Low,0.234,Low,Mobility Limitation
3,P00004,45 to 49 years,Urban,1,0.2000,Low,0.096,Low,0.1420,Low,0.106,Low,Hypertension
4,P00005,15 to 19 years,Urban,0,0.1200,Low,0.116,Low,0.0120,Low,0.116,Low,Mobility Limitation
5,P00006,15 to 19 years,Urban,0,0.0200,Low,0.016,Low,0.0120,Low,0.016,Low,Age
6,P00007,5 to 9 years,Rural,0,0.2125,Low,0.160,Low,0.3575,Medium,0.260,Low,Mobility Limitation
7,P00008,65 to 69 years,Urban,0,0.1450,Low,0.166,Low,0.1870,Low,0.166,Low,Age
8,P00009,45 to 49 years,Urban,0,0.1700,Low,0.256,Low,0.4420,Medium,0.256,Low,Low Digital Access
9,P00010,55 to 59 years,Urban,0,0.1050,Low,0.084,Low,0.0630,Low,0.084,Low,Age


In [ ]:
# ____________________________________________________________
# Review the overall distribution of risk levels
# ____________________________________________________________
print(risk_df["ed_escalation_risk_level"].value_counts(normalize=True))
print(risk_df["readmission_vulnerability_level"].value_counts(normalize=True))
print(risk_df["avoidable_utilization_risk_level"].value_counts(normalize=True))
print(risk_df["care_complexity_level"].value_counts(normalize=True))

# The system shows how risk is spread across the entire population***

ed_escalation_risk_level
Low       0.6885
Medium    0.2767
High      0.0348
Name: proportion, dtype: float64
readmission_vulnerability_level
Low       0.7836
Medium    0.2067
High      0.0097
Name: proportion, dtype: float64
avoidable_utilization_risk_level
Low       0.6297
Medium    0.3330
High      0.0373
Name: proportion, dtype: float64
care_complexity_level
Low       0.7982
Medium    0.1990
High      0.0028
Name: proportion, dtype: float64


In [ ]:
# ____________________________________________________________
# Review average risk scores for sanity checking
# ____________________________________________________________
print("Average ED escalation risk:", risk_df["ed_escalation_risk_score"].mean())
print("Average readmission vulnerability:", risk_df["readmission_vulnerability_score"].mean())
print("Average avoidable utilization risk:", risk_df["avoidable_utilization_risk_score"].mean())
print("Average care complexity score:", risk_df["care_complexity_score"].mean())

# The system calculates overall average risk levels for the population***
# act as a baseline that can later be compared when testing
# different scenarios or policies.

Average ED escalation risk: 0.2637703770377038
Average readmission vulnerability: 0.21854005400540055
Average avoidable utilization risk: 0.29103450345034504
Average care complexity score: 0.2208242824282428


In [ ]:
# ____________________________________________________________
# Save the risk-scored synthetic population
# Can quantify healthcare risk on patient and population level***
# ____________________________________________________________
risk_df.to_csv("synthetic_patients_step5_risk_scored.csv", index=False)


### What this section does
This step transforms the ***synthetic population into a risk-aware population that is usable enabling healthcare decesion making in the model***.

Each synthetic patient receives a **planning-level risk profile** across four key dimensions:

- **ED escalation risk**
- **Readmission vulnerability**
- **Avoidable utilization risk**
- **Care complexity**

These scores are then categorized into **Low, Medium, or High risk levels**, and the system also identifies a **top risk driver** for each patient.

After this stage, the patients are no longer just synthetic records — they become **operationally meaningful synthetic healthcare entities**.

### Why this step exists
The goal of the project is not only to generate synthetic data, but to demonstrate how synthetic data can support **healthcare planning and innovation**.

This step adds a **predictive planning layer** that makes the synthetic population useful for:

- **Care routing decisions**
- **Healthcare policy simulation**
- **System-level decision support**

*It enables smarter care routing, better system simulation, and more informed policy testing.*

*where the system learns who needs care most and why*

### Technologies used
This stage relies on transparent and explainable scoring methods, including:

- **Weighted rule-based risk scoring**
- **Feature mapping from patient attributes**
- **Threshold-based risk categorization**
- **Simple explainability logic to identify top risk drivers**

This approach is intentionally **interpretable and practical**, making it:

- **Easy to understand**
- **Fast to compute**
- **Defensible during evaluation**
- **Appropriate for a planning-focused synthetic system**

### Why this matters
This is the stage where the synthetic population becomes a **functional planning model**.

Before this step, the system only contains **synthetic patient records**.  
After this step, the system now has:

- **Risk-scored patient profiles**
- **Explainable patient risk drivers**
- **Inputs for care routing**
- **Inputs for healthcare scenario simulations**

These outputs enable the downstream components of the system.

**This block assigns explainable risk scores to each synthetic patient.**  
The system evaluates factors such as age, chronic disease burden, and access barriers to estimate ED escalation risk, readmission vulnerability, avoidable utilization risk, and overall care complexity.



# 6. Distributed Care Navigation and Patient Routing Engine

##### This step turns risk scores into real care decisions. The system assigns every patient to the most appropriate care pathway. Uses simple rule-based logic to match patient risk to care options.

In [ ]:
# ____________________________________________________________
# Purpose:
# Take the synthetic, risk-scored patient population from
# 'Predictive Risk Analysis and Patient Risk Scoring'
# and route each patient to the most appropriate care setting
# within a Southlake-like Distributed Health Network.
#
# simulates how patients would actually flow through healthcare system ***
#
# This step acts like a "Care GPS" by assigning patients to:
# - Emergency Department
# - OHT / Primary Care Clinic
# - Virtual Urgent Care
# - Southlake@home
# - Remote Monitoring / Telehomecare
# - Mobile / Community Outreach
# - Mental Health Assessment Centre
#
# It also explains the route and flags potential equity issues.
# ____________________________________________________________

# ____________________________________________________________
# Create a working copy of the risk-scored dataset
# - Protects 'Predictive Risk Analysis and
#   Patient Risk Scoring' outputs from being overwritten.
# ____________________________________________________________

navigator_df = risk_df.copy()


In [ ]:
# ____________________________________________________________
# Define the care navigation function
# - This function determines the most appropriate care setting
#   for a patient based on risk, barriers, and need.
# ____________________________________________________________
def assign_care_route(row):

    # Mental health patients with meaningful utilization risk
    if row["mental_health_flag"] == 1 and row["avoidable_utilization_risk_level"] in ["Medium", "High"]:
        return "Mental Health Assessment Centre"

    # Patients with the highest ED escalation risk
    if row["ed_escalation_risk_level"] == "High":
        return "Emergency Department"

    # Frail, complex, mobility-limited patients with barriers
    if (
        row["care_complexity_level"] == "High"
        or (
            row["age_risk_score"] > 0.7
            and row["mobility_limitation"] == 1
            and row["transport_barrier"] == 1
        )
    ):
        return "Southlake@home"

    # Rural patients with transportation barriers
    if row["rurality"] == "Rural" and row["transport_barrier"] == 1:
        return "Mobile / Community Outreach"

    # Medium avoidable-risk patients with strong digital access
    if (
        row["avoidable_utilization_risk_level"] in ["Medium", "High"]
        and row["digital_access"] == "High"
        and row["ed_escalation_risk_level"] != "High"
    ):
        return "Virtual Urgent Care"

    # Chronic patients who are not acutely unstable
    if (
        row["chronic_condition_count"] >= 2
        and row["ed_escalation_risk_level"] in ["Low", "Medium"]
    ):
        return "Remote Monitoring / Telehomecare"

    # Default pathway for lower-acuity cases
    return "OHT / Primary Care Clinic"


In [ ]:
# ____________________________________________________________
# Apply care route assignment to every synthetic patient
# ____________________________________________________________
navigator_df["recommended_care_route"] = navigator_df.apply(assign_care_route, axis=1)

# Preview the first few routed patients
navigator_df[[
    "patient_id",
    "age_group",
    "rurality",
    "ed_escalation_risk_level",
    "readmission_vulnerability_level",
    "avoidable_utilization_risk_level",
    "care_complexity_level",
    "mental_health_flag",
    "recommended_care_route"
]].head(10)

# Each patient now has a recommended care route based on ***

,patient_id,age_group,rurality,ed_escalation_risk_level,readmission_vulnerability_level,avoidable_utilization_risk_level,care_complexity_level,mental_health_flag,recommended_care_route
0,P00001,30 to 34 years,Urban,Low,Low,Medium,Medium,1,Mental Health Assessment Centre
1,P00002,75 to 79 years,Urban,Medium,Low,Medium,Low,0,Virtual Urgent Care
2,P00003,55 to 59 years,Urban,Medium,Low,Low,Low,0,OHT / Primary Care Clinic
3,P00004,45 to 49 years,Urban,Low,Low,Low,Low,0,OHT / Primary Care Clinic
4,P00005,15 to 19 years,Urban,Low,Low,Low,Low,0,OHT / Primary Care Clinic
5,P00006,15 to 19 years,Urban,Low,Low,Low,Low,0,OHT / Primary Care Clinic
6,P00007,5 to 9 years,Rural,Low,Low,Medium,Low,0,Mobile / Community Outreach
7,P00008,65 to 69 years,Urban,Low,Low,Low,Low,0,OHT / Primary Care Clinic
8,P00009,45 to 49 years,Urban,Low,Low,Medium,Low,0,OHT / Primary Care Clinic
9,P00010,55 to 59 years,Urban,Low,Low,Low,Low,0,OHT / Primary Care Clinic


In [ ]:
# ____________________________________________________________
# Review how the synthetic population is distributed across
# care destinations.
# ____________________________________________________________
print(navigator_df["recommended_care_route"].value_counts())
print(navigator_df["recommended_care_route"].value_counts(normalize=True))

# Now routed to primary care, small number go emergency care***

recommended_care_route
OHT / Primary Care Clinic           6897
Remote Monitoring / Telehomecare     839
Mobile / Community Outreach          809
Mental Health Assessment Centre      609
Virtual Urgent Care                  545
Emergency Department                 256
Southlake@home                        45
Name: count, dtype: int64
recommended_care_route
OHT / Primary Care Clinic           0.6897
Remote Monitoring / Telehomecare    0.0839
Mobile / Community Outreach         0.0809
Mental Health Assessment Centre     0.0609
Virtual Urgent Care                 0.0545
Emergency Department                0.0256
Southlake@home                      0.0045
Name: proportion, dtype: float64


In [ ]:
# ____________________________________________________________
# Explain why each patient was routed to that setting
# - Makes the navigator more interpretable and useful
# ____________________________________________________________
def explain_route(row):
    route = row["recommended_care_route"]

    if route == "Mental Health Assessment Centre":
        return "Routed due to mental health needs and elevated avoidable utilization risk."

    elif route == "Emergency Department":
        return "Routed due to high ED escalation risk."

    elif route == "Southlake@home":
        return "Routed due to high care complexity, age-related fragility, and access barriers."

    elif route == "Mobile / Community Outreach":
        return "Routed due to rural location and transportation barriers."

    elif route == "Virtual Urgent Care":
        return "Routed due to moderate avoidable utilization risk and strong digital access."

    elif route == "Remote Monitoring / Telehomecare":
        return "Routed due to chronic disease burden without acute escalation."

    else:
        return "Routed to primary care as the lowest-acuity and most appropriate setting."

# Apply route explanation
navigator_df["route_explanation"] = navigator_df.apply(explain_route, axis=1)

# Preview the explanations
navigator_df[[
    "patient_id",
    "recommended_care_route",
    "route_explanation"
]].head(10)

# each decision has a explanation ***

,patient_id,recommended_care_route,route_explanation
0,P00001,Mental Health Assessment Centre,Routed due to mental health needs and elevated...
1,P00002,Virtual Urgent Care,Routed due to moderate avoidable utilization r...
2,P00003,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...
3,P00004,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...
4,P00005,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...
5,P00006,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...
6,P00007,Mobile / Community Outreach,Routed due to rural location and transportatio...
7,P00008,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...
8,P00009,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...
9,P00010,OHT / Primary Care Clinic,Routed to primary care as the lowest-acuity an...


In [ ]:
# ____________________________________________________________
# Create a simple equity warning system
# - This flags cases where care routing may not fit the patient's
#   access context.
# ____________________________________________________________
def equity_warning(row):
    if row["recommended_care_route"] == "Virtual Urgent Care" and row["digital_access"] == "Low":
        return "Digital access mismatch"

    if row["recommended_care_route"] == "OHT / Primary Care Clinic" and row["transport_barrier"] == 1:
        return "Transportation barrier"

    if row["recommended_care_route"] == "Mobile / Community Outreach" and row["rurality"] == "Urban":
        return "Possible outreach mismatch"

    return "None"

# Apply equity warnings
navigator_df["equity_warning"] = navigator_df.apply(equity_warning, axis=1)

# Review equity warnings
print(navigator_df["equity_warning"].value_counts())

# High risk patients are sent to more intensive care


equity_warning
None                      8910
Transportation barrier    1090
Name: count, dtype: int64


In [ ]:
# ____________________________________________________________
# Check whether the routing logic behaves sensibly
# by comparing routes across ED risk levels.
# ____________________________________________________________
pd.crosstab(
    navigator_df["ed_escalation_risk_level"],
    navigator_df["recommended_care_route"],
    normalize="index"
)



recommended_care_route,Emergency Department,Mental Health Assessment Centre,Mobile / Community Outreach,OHT / Primary Care Clinic,Remote Monitoring / Telehomecare,Southlake@home,Virtual Urgent Care
ed_escalation_risk_level,,,,,,,
High,0.735632,0.264368,0.000000,0.000000,0.000000,0.000000,0.000000
Low,0.000000,0.023675,0.071895,0.874946,0.008860,0.000000,0.020625
Medium,0.000000,0.127936,0.113480,0.315504,0.281171,0.016263,0.145645


In [ ]:
# ____________________________________________________________
# Check whether rural and urban patients are routed
# differently, as expected.
# ____________________________________________________________
pd.crosstab(
    navigator_df["rurality"],
    navigator_df["recommended_care_route"],
    normalize="index"
)

# Rural patients are more often sent to outreach services, ***
# while urban patients rely more on clinics.

recommended_care_route,Emergency Department,Mental Health Assessment Centre,Mobile / Community Outreach,OHT / Primary Care Clinic,Remote Monitoring / Telehomecare,Southlake@home,Virtual Urgent Care
rurality,,,,,,,
Rural,0.043700,0.094779,0.459137,0.316118,0.035187,0.010216,0.040863
Urban,0.021729,0.053654,0.000000,0.769604,0.094319,0.003277,0.057417


In [ ]:
# ____________________________________________________________
# 9. Save the routed synthetic patient dataset
# ____________________________________________________________

navigator_df.to_csv("synthetic_patients_step6_routed.csv", index=False)


### What this section does
This step converts the **synthetic, risk-scored patient population into a care-routing simulation population**.

The system functions like a **planning-focused Care GPS**, determining the most appropriate care pathway for each synthetic patient.

For every patient, the agent determines:

- **Recommended care destination**
- **Reason for the routing decision**
- **Potential equity concerns in access to care**

After this stage, the system now includes:

- **Risk-scored synthetic patients**
- **Assigned care destinations**
- **Explainable routing decisions**
- **Equity warning flags**

### Why this step exists
The goal of the project is not only to generate synthetic healthcare populations, but to demonstrate how those populations can be used to **test distributed care strategies**.

This step connects the synthetic population to the **Distributed Health Network model**, simulating how patients could be routed across care settings such as:

- **Hospital care**
- **Home-based care**
- **Community outreach services**
- **Virtual care**
- **Mental health diversion pathways**
- **Primary care**

*This step turns the model into a real care navigation system. This is what enables system-level planning and optimization.*

### Technologies used
This stage uses transparent decision logic to simulate care pathways, including:

- **Rule-based routing logic**
- **Explainability functions**
- **Equity and fairness checks**
- **Cross-tab validation of routing outcomes**

This approach is intentionally:

- **Interpretable**
- **Clinically intuitive**
- **Easy to explain in presentations**
- **Relevant for healthcare policy and planning**

*It helps reduce emergency department pressure and improve access to care.*



### Why this matters
This step is where the project becomes **health system–specific and operational**.

Before this stage, the system contains **risk-scored synthetic patients**.  
After this stage, the system simulates a **Distributed Health Network**, producing:

- **Patient flow across care settings**
- **Explainable care recommendations**
- **Equity-aware navigation decisions**

This significantly increases the realism and practical value of the simulation.

**This block simulates how synthetic patients move through the healthcare system.**  
Based on each patient’s risk profile and access barriers, the system recommends the most appropriate care setting while also flagging potential equity concerns.

*This is where the system decides the right care, in the right place, at the right time.*

# 7. Scenario Simulation Engine

##### This step simulates how different healthcare policies change the system. The system runs multiple policy scenarios on the same synthetic population.

*It copies the dataset, changes routing rules, and compares results to baseline. It turns the model into a decision-making and planning tool.*

In [ ]:
# ____________________________________________________________
# Purpose:
# Take the routed synthetic patient population from 6 and
# simulate how different policy strategies would change demand
# across the Southlake-like Distributed Health Network.
#
# This step compares multiple scenarios, such as:
# - Baseline
# - Expanded Virtual Urgent Care
# - Expanded Southlake@home + Outreach
# - Equity-Aware Routing
#
# It then measures how each scenario changes:
# - ED demand
# - primary care demand
# - virtual care demand
# - home-based care demand
# - outreach demand
# - equity warning counts
#
# Finally, it identifies which policy performs best for
# different system goals.
# ____________________________________________________________

# ____________________________________________________________
# Create a working copy of the routed synthetic population
# - This represents the baseline simulated health system
# ____________________________________________________________

scenario_df = navigator_df.copy()


In [ ]:
# ____________________________________________________________
# Create a reusable summary function
# - Converts a routed scenario into system-level output metrics
# ____________________________________________________________
def summarize_scenario(df, scenario_name):

    # Count how many patients go to each route
    route_counts = df["recommended_care_route"].value_counts()

    # Count equity warning types
    equity_counts = df["equity_warning"].value_counts()

    # Build a one-row scenario summary table
    summary = {
        "scenario": scenario_name,
        "total_patients": len(df),
        "ed_count": route_counts.get("Emergency Department", 0),
        "primary_care_count": route_counts.get("OHT / Primary Care Clinic", 0),
        "virtual_urgent_care_count": route_counts.get("Virtual Urgent Care", 0),
        "southlake_home_count": route_counts.get("Southlake@home", 0),
        "remote_monitoring_count": route_counts.get("Remote Monitoring / Telehomecare", 0),
        "mental_health_count": route_counts.get("Mental Health Assessment Centre", 0),
        "outreach_count": route_counts.get("Mobile / Community Outreach", 0),
        "transport_barrier_warnings": equity_counts.get("Transportation barrier", 0),
        "digital_access_mismatch_warnings": equity_counts.get("Digital access mismatch", 0)
    }

    return pd.DataFrame([summary])

# Summarize the baseline scenario
baseline_summary = summarize_scenario(scenario_df, "Baseline")
baseline_summary

# shows the original system behavior before any changes (reference)***

,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings
0,Baseline,10000,256,6897,545,45,839,609,809,1090,0


In [ ]:
# ____________________________________________________________
# Scenario B: Expanded Virtual Urgent Care
# Policy logic:
# - Move more avoidable / digitally connected patients into
#   virtual urgent care instead of primary care or ED.
# ____________________________________________________________
scenario_b_df = navigator_df.copy()

virtual_shift_mask = (
    (scenario_b_df["avoidable_utilization_risk_level"].isin(["Medium", "High"])) &
    (scenario_b_df["ed_escalation_risk_level"].isin(["Low", "Medium"])) &
    (scenario_b_df["digital_access"].isin(["Medium", "High"])) &
    (scenario_b_df["recommended_care_route"].isin([
        "OHT / Primary Care Clinic",
        "Emergency Department",
        "Remote Monitoring / Telehomecare"
    ]))
)

scenario_b_df.loc[virtual_shift_mask, "recommended_care_route"] = "Virtual Urgent Care"
scenario_b_df.loc[virtual_shift_mask, "route_explanation"] = "Routed under expanded virtual urgent care policy."

# Refresh equity warnings after rerouting
scenario_b_df["equity_warning"] = scenario_b_df.apply(equity_warning, axis=1)

# Summarize Scenario B
scenario_b_summary = summarize_scenario(scenario_b_df, "Expanded Virtual Urgent Care")
scenario_b_summary

# Virtual care increases significantly but ED stays the same ***
# Expanding virtual care shifts demand, but doesn’t always reduce ED


,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings
0,Expanded Virtual Urgent Care,10000,256,6226,1397,45,658,609,809,849,0


In [ ]:
# ____________________________________________________________
# Scenario C: Expanded Southlake@home + Outreach
# Policy logic:
# - Move more frail, barrier-heavy, and rural patients into
#   home-based and outreach care.
# ____________________________________________________________
scenario_c_df = navigator_df.copy()

# Shift more complex, mobility-limited patients to Southlake@home
southlake_home_mask = (
    (scenario_c_df["care_complexity_level"].isin(["Medium", "High"])) &
    (scenario_c_df["mobility_limitation"] == 1) &
    (scenario_c_df["transport_barrier"] == 1) &
    (scenario_c_df["recommended_care_route"].isin([
        "OHT / Primary Care Clinic",
        "Emergency Department",
        "Remote Monitoring / Telehomecare"
    ]))
)

scenario_c_df.loc[southlake_home_mask, "recommended_care_route"] = "Southlake@home"
scenario_c_df.loc[southlake_home_mask, "route_explanation"] = "Routed under expanded Southlake@home policy."

# Shift more rural barrier patients to outreach
outreach_mask = (
    (scenario_c_df["rurality"] == "Rural") &
    (scenario_c_df["transport_barrier"] == 1) &
    (scenario_c_df["recommended_care_route"].isin([
        "OHT / Primary Care Clinic",
        "Remote Monitoring / Telehomecare"
    ]))
)

scenario_c_df.loc[outreach_mask, "recommended_care_route"] = "Mobile / Community Outreach"
scenario_c_df.loc[outreach_mask, "route_explanation"] = "Routed under expanded outreach policy."

# Refresh equity warnings
scenario_c_df["equity_warning"] = scenario_c_df.apply(equity_warning, axis=1)

# Summarize Scenario C
scenario_c_summary = summarize_scenario(scenario_c_df, "Expanded Southlake@home + Outreach")
scenario_c_summary

# Emergency Department visits decrease the most in this scenario ***
# More patients are handled through home-based care (pressure)


,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings
0,Expanded Southlake@home + Outreach,10000,197,6825,545,183,832,609,809,1018,0


In [ ]:
# ____________________________________________________________
# Create Scenario D: Equity-Aware Routing
# Policy logic:
# - Avoid routing low-digital-access patients into virtual care,
#   and redirect barrier-heavy patients toward outreach instead
#   of less accessible settings.
# ____________________________________________________________
scenario_d_df = navigator_df.copy()

# Move low-digital-access patients out of virtual care
equity_mask_virtual = (
    (scenario_d_df["recommended_care_route"] == "Virtual Urgent Care") &
    (scenario_d_df["digital_access"] == "Low")
)

scenario_d_df.loc[equity_mask_virtual, "recommended_care_route"] = "OHT / Primary Care Clinic"
scenario_d_df.loc[equity_mask_virtual, "route_explanation"] = "Re-routed under equity-aware policy due to low digital access."

# Move transport-barrier patients from primary care to outreach
equity_mask_transport = (
    (scenario_d_df["recommended_care_route"] == "OHT / Primary Care Clinic") &
    (scenario_d_df["transport_barrier"] == 1)
)

scenario_d_df.loc[equity_mask_transport, "recommended_care_route"] = "Mobile / Community Outreach"
scenario_d_df.loc[equity_mask_transport, "route_explanation"] = "Re-routed under equity-aware policy due to transportation barriers."

# Refresh equity warnings
scenario_d_df["equity_warning"] = scenario_d_df.apply(equity_warning, axis=1)

# Summarize Scenario D
scenario_d_summary = summarize_scenario(scenario_d_df, "Equity-Aware Routing")
scenario_d_summary

# Access barriers are removed and outreach increases significantly***
# improves fairness in the system, even if it doesn’t reduce ED visits

,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings
0,Equity-Aware Routing,10000,256,5807,545,45,839,609,1899,0,0


In [ ]:
# ____________________________________________________________
# Combine all scenario summaries into one comparison table
# ____________________________________________________________
scenario_comparison = pd.concat(
    [baseline_summary, scenario_b_summary, scenario_c_summary, scenario_d_summary],
    ignore_index=True
)

scenario_comparison

# scenarios are compared side-by-side (trade-offs) ***

,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings
0,Baseline,10000,256,6897,545,45,839,609,809,1090,0
1,Expanded Virtual Urgent Care,10000,256,6226,1397,45,658,609,809,849,0
2,Expanded Southlake@home + Outreach,10000,197,6825,545,183,832,609,809,1018,0
3,Equity-Aware Routing,10000,256,5807,545,45,839,609,1899,0,0


In [ ]:
# ____________________________________________________________
# Calculate differences from baseline
# - Makeing policy impact easier to interpret
# ____________________________________________________________
baseline_row = scenario_comparison.iloc[0]

comparison_with_deltas = scenario_comparison.copy()

for col in [
    "ed_count",
    "primary_care_count",
    "virtual_urgent_care_count",
    "southlake_home_count",
    "remote_monitoring_count",
    "mental_health_count",
    "outreach_count",
    "transport_barrier_warnings",
    "digital_access_mismatch_warnings"
]:
    comparison_with_deltas[f"{col}_delta_vs_baseline"] = (
        comparison_with_deltas[col] - baseline_row[col]
    )

comparison_with_deltas

# Shows exactly how each policy changes the system ***

,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings,ed_count_delta_vs_baseline,primary_care_count_delta_vs_baseline,virtual_urgent_care_count_delta_vs_baseline,southlake_home_count_delta_vs_baseline,remote_monitoring_count_delta_vs_baseline,mental_health_count_delta_vs_baseline,outreach_count_delta_vs_baseline,transport_barrier_warnings_delta_vs_baseline,digital_access_mismatch_warnings_delta_vs_baseline
0,Baseline,10000,256,6897,545,45,839,609,809,1090,0,0,0,0,0,0,0,0,0,0
1,Expanded Virtual Urgent Care,10000,256,6226,1397,45,658,609,809,849,0,0,-671,852,0,-181,0,0,-241,0
2,Expanded Southlake@home + Outreach,10000,197,6825,545,183,832,609,809,1018,0,-59,-72,0,138,-7,0,0,-72,0
3,Equity-Aware Routing,10000,256,5807,545,45,839,609,1899,0,0,0,-1090,0,0,0,0,1090,-1090,0


In [ ]:
# ____________________________________________________________
# Create a simple policy recommendation engine
# - Identifies which scenario performs best for key goals ***
# ____________________________________________________________
def policy_recommendation(df):
    best_ed_reduction = df.loc[df["ed_count_delta_vs_baseline"].idxmin(), "scenario"]
    best_equity = df.loc[df["transport_barrier_warnings_delta_vs_baseline"].idxmin(), "scenario"]
    best_virtual_shift = df.loc[df["virtual_urgent_care_count_delta_vs_baseline"].idxmax(), "scenario"]
    best_home_shift = df.loc[df["southlake_home_count_delta_vs_baseline"].idxmax(), "scenario"]

    print("Policy Recommendation Summary")
    print("--------------------------------")
    print("Best for reducing ED burden:", best_ed_reduction)
    print("Best for reducing transport-barrier warnings:", best_equity)
    print("Best for increasing virtual care use:", best_virtual_shift)
    print("Best for increasing home-based care:", best_home_shift)

# Run the recommendation engine
policy_recommendation(comparison_with_deltas)


Policy Recommendation Summary
--------------------------------
Best for reducing ED burden: Expanded Southlake@home + Outreach
Best for reducing transport-barrier warnings: Equity-Aware Routing
Best for increasing virtual care use: Expanded Virtual Urgent Care
Best for increasing home-based care: Expanded Southlake@home + Outreach


In [ ]:
# ____________________________________________________________
# Show the scenario selected in the Step 0 control panel
# ____________________________________________________________
selected_scenario_name = agent_config.get("scenario_choice", "Baseline")

scenario_lookup = {
    "Baseline": baseline_summary,
    "Expanded Virtual Urgent Care": scenario_b_summary,
    "Expanded Southlake@home + Outreach": scenario_c_summary,
    "Equity-Aware Routing": scenario_d_summary
}

selected_scenario_summary = scenario_lookup[selected_scenario_name]

print("Selected Scenario Summary:")
display(selected_scenario_summary)


Selected Scenario Summary:


,scenario,total_patients,ed_count,primary_care_count,virtual_urgent_care_count,southlake_home_count,remote_monitoring_count,mental_health_count,outreach_count,transport_barrier_warnings,digital_access_mismatch_warnings
0,Baseline,10000,256,6897,545,45,839,609,809,1090,0


In [ ]:
# ____________________________________________________________
# Save all major scenario outputs
# ____________________________________________________________
scenario_comparison.to_csv("scenario_comparison_step7.csv", index=False)
comparison_with_deltas.to_csv("scenario_comparison_with_deltas_step7.csv", index=False)
scenario_b_df.to_csv("scenario_b_expanded_virtual.csv", index=False)
scenario_c_df.to_csv("scenario_c_home_outreach.csv", index=False)
scenario_d_df.to_csv("scenario_d_equity_aware.csv", index=False)


In [ ]:
# ____________________________________________________________
# Display a compact, presentation-friendly comparison view
# ____________________________________________________________
comparison_with_deltas[[
    "scenario",
    "ed_count",
    "ed_count_delta_vs_baseline",
    "virtual_urgent_care_count",
    "virtual_urgent_care_count_delta_vs_baseline",
    "southlake_home_count",
    "southlake_home_count_delta_vs_baseline",
    "outreach_count",
    "outreach_count_delta_vs_baseline",
    "transport_barrier_warnings",
    "transport_barrier_warnings_delta_vs_baseline"
]]


,scenario,ed_count,ed_count_delta_vs_baseline,virtual_urgent_care_count,virtual_urgent_care_count_delta_vs_baseline,southlake_home_count,southlake_home_count_delta_vs_baseline,outreach_count,outreach_count_delta_vs_baseline,transport_barrier_warnings,transport_barrier_warnings_delta_vs_baseline
0,Baseline,256,0,545,0,45,0,809,0,1090,0
1,Expanded Virtual Urgent Care,256,0,1397,852,45,0,809,0,849,-241
2,Expanded Southlake@home + Outreach,197,-59,545,0,183,138,809,0,1018,-72
3,Equity-Aware Routing,256,0,545,0,45,0,1899,1090,0,-1090


## Step 7 — Policy Scenario Simulation

### What this section does
This step functions as a **digital twin policy simulator** for the healthcare system.

Using the routed synthetic patient population, the system evaluates **how different policy choices change system demand and care access**.

The simulation tests multiple strategies and measures their impact on:

- **Emergency Department (ED) burden**
- **Primary care demand**
- **Virtual care utilization**
- **Home-based care utilization**
- **Community outreach usage**
- **Equity warning flags**

The system then compares outcomes across scenarios and identifies **which strategy performs best for different strategic goals**.

### Why this step exists
The purpose of the project is not only to generate synthetic healthcare data, but to demonstrate how that data can support:

- **Healthcare planning**
- **Policy experimentation**
- **System redesign**
- **Privacy-safe innovation**

This step shows how a synthetic population can act as a **digital twin of the health system**, allowing decision-makers to test strategies without using real patient data.

*Scenarios let us test different healthcare strategies to see how they would change patient flow and system performance before applying them in real life.*

### Technologies used
This stage relies on transparent simulation methods, including:

- **Scenario-based policy simulation**
- **Policy rule reassignment**
- **System-level outcome summarization**
- **Baseline vs. scenario comparison**
- **Simple policy recommendation logic**

These methods are intentionally:

- **Interpretable**
- **Operationally meaningful**
- **Easy to present**
- **Aligned with health system planning goals**

*They are created by copying the same synthetic population, changing routing rules (like expanding virtual care), and comparing results to the baseline.*

### Why this matters
This step transforms the project from a **synthetic data generator** into a **synthetic health system policy simulator**.

*This helps identify which policies actually improve outcomes, reduce pressure, or fix access issues—turning data into clear, actionable decisions.*

Before this stage, the system contains:

- Synthetic patients  
- Risk scores  
- Care routing recommendations  

After this stage, the system produces:

- **Baseline system behavior**
- **Multiple policy alternatives**
- **Measured system impacts**
- **Policy recommendations**

This is what makes the project **decision-useful for healthcare planning**.

**THis block uses the synthetic population as a digital twin of the health system.**  
The agent simulates how different strategies — such as expanding virtual care, home-based care, or equity-aware routing — affect system demand and access to care.

# 8. Hypothetical Patient Matching and Care Route Demo

In [ ]:
# ____________________________________________________________
# Purpose:
# Allow the user to enter a hypothetical patient profile,
# compare it against the synthetic population, and return:
# - the most similar synthetic patients
# - average risk scores among those matches
# - the most common recommended care route
# - the most common top risk driver
#
# Important:
# This is NOT a clinical diagnosis or real triage tool.
# It is a synthetic planning / navigation demonstration layer.
# ____________________________________________________________

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# ____________________________________________________________
# Make sure we use the fully processed patient dataset
# navigator_df already contains:
# - synthetic patient features
# - risk scores
# - care routes
# - risk drivers
# - equity warnings
# ____________________________________________________________
patient_match_df = navigator_df.copy()

# ____________________________________________________________
# Create input widgets for a hypothetical patient
# - Represent a "what if" patient profile
# ____________________________________________________________
demo_age_group = widgets.Dropdown(
    options=sorted(patient_match_df["age_group"].dropna().unique().tolist()),
    description="Age Group:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

demo_sex = widgets.Dropdown(
    options=["Female", "Male"],
    description="Sex:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_rurality = widgets.Dropdown(
    options=["Urban", "Rural"],
    description="Rurality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_diabetes = widgets.Dropdown(
    options=[0, 1],
    description="Diabetes:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_hypertension = widgets.Dropdown(
    options=[0, 1],
    description="Hypertension:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_copd = widgets.Dropdown(
    options=[0, 1],
    description="COPD:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_heart = widgets.Dropdown(
    options=[0, 1],
    description="Heart Disease:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_mental = widgets.Dropdown(
    options=[0, 1],
    description="Mental Health:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

demo_digital = widgets.Dropdown(
    options=["Low", "Medium", "High"],
    description="Digital Access:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

demo_primary_care = widgets.Dropdown(
    options=["Good", "Limited"],
    description="Primary Care Access:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px")
)

demo_mobility = widgets.Dropdown(
    options=[0, 1],
    description="Mobility Limitation:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

demo_transport = widgets.Dropdown(
    options=[0, 1],
    description="Transport Barrier:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px")
)

demo_n_matches = widgets.IntSlider(
    value=25,
    min=5,
    max=100,
    step=5,
    description="Top Matches:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

match_button = widgets.Button(
    description="Find Similar Synthetic Patients",
    button_style="success",
    icon="search"
)

match_output = widgets.Output()

# ____________________________________________________________
# Define the feature comparison logic
# We compare the hypothetical patient to all synthetic patients.
#
# The approach used here is a simple weighted similarity score:
# - exact matches on categorical/binary features add points
# - age group match adds extra weight
# - access barriers also contribute
# ____________________________________________________________
def compute_similarity_scores(df, patient_profile):
    temp_df = df.copy()

    # Start all patients with 0 similarity score
    temp_df["similarity_score"] = 0.0

    # Weighted exact-match scoring
    temp_df["similarity_score"] += np.where(temp_df["age_group"] == patient_profile["age_group"], 3.0, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["sex"] == patient_profile["sex"], 1.0, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["rurality"] == patient_profile["rurality"], 1.5, 0.0)

    temp_df["similarity_score"] += np.where(temp_df["diabetes_flag"] == patient_profile["diabetes_flag"], 1.5, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["hypertension_flag"] == patient_profile["hypertension_flag"], 1.5, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["copd_flag"] == patient_profile["copd_flag"], 1.5, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["heart_disease_flag"] == patient_profile["heart_disease_flag"], 1.5, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["mental_health_flag"] == patient_profile["mental_health_flag"], 1.5, 0.0)

    temp_df["similarity_score"] += np.where(temp_df["digital_access"] == patient_profile["digital_access"], 1.0, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["primary_care_access"] == patient_profile["primary_care_access"], 1.0, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["mobility_limitation"] == patient_profile["mobility_limitation"], 1.0, 0.0)
    temp_df["similarity_score"] += np.where(temp_df["transport_barrier"] == patient_profile["transport_barrier"], 1.0, 0.0)

    return temp_df

# ____________________________________________________________
# Define what happens when the user clicks the match button
# ____________________________________________________________
def on_match_clicked(button):
    with match_output:
        clear_output()

        # Build the hypothetical patient profile
        patient_profile = {
            "age_group": demo_age_group.value,
            "sex": demo_sex.value,
            "rurality": demo_rurality.value,
            "diabetes_flag": demo_diabetes.value,
            "hypertension_flag": demo_hypertension.value,
            "copd_flag": demo_copd.value,
            "heart_disease_flag": demo_heart.value,
            "mental_health_flag": demo_mental.value,
            "digital_access": demo_digital.value,
            "primary_care_access": demo_primary_care.value,
            "mobility_limitation": demo_mobility.value,
            "transport_barrier": demo_transport.value
        }

        # Score all synthetic patients for similarity
        scored_df = compute_similarity_scores(patient_match_df, patient_profile)

        # Pull top-N most similar synthetic patients
        top_matches = scored_df.sort_values(
            by="similarity_score",
            ascending=False
        ).head(demo_n_matches.value)

        # Summarize matched cohort
        avg_ed_score = top_matches["ed_escalation_risk_score"].mean()
        avg_readmit_score = top_matches["readmission_vulnerability_score"].mean()
        avg_avoidable_score = top_matches["avoidable_utilization_risk_score"].mean()
        avg_complexity_score = top_matches["care_complexity_score"].mean()

        most_common_route = top_matches["recommended_care_route"].mode().iloc[0]
        most_common_driver = top_matches["top_risk_driver"].mode().iloc[0]
        most_common_ed_level = top_matches["ed_escalation_risk_level"].mode().iloc[0]
        most_common_readmit_level = top_matches["readmission_vulnerability_level"].mode().iloc[0]
        most_common_avoidable_level = top_matches["avoidable_utilization_risk_level"].mode().iloc[0]
        most_common_complexity_level = top_matches["care_complexity_level"].mode().iloc[0]

        # Display results
        display(Markdown("## Hypothetical Patient Profile"))
        print(patient_profile)

        display(Markdown("## Best-Matching Synthetic Patient Summary"))
        print(f"Top match cohort size: {len(top_matches)}")
        print(f"Most common recommended care route: {most_common_route}")
        print(f"Most common top risk driver: {most_common_driver}")
        print(f"Typical ED escalation level: {most_common_ed_level}")
        print(f"Typical readmission vulnerability level: {most_common_readmit_level}")
        print(f"Typical avoidable utilization level: {most_common_avoidable_level}")
        print(f"Typical care complexity level: {most_common_complexity_level}")

        display(Markdown("## Average Risk Scores of Similar Synthetic Patients"))
        print(f"Average ED escalation risk score: {avg_ed_score:.3f}")
        print(f"Average readmission vulnerability score: {avg_readmit_score:.3f}")
        print(f"Average avoidable utilization risk score: {avg_avoidable_score:.3f}")
        print(f"Average care complexity score: {avg_complexity_score:.3f}")

        display(Markdown("## Top Matching Synthetic Patients (sample)"))
        display(
            top_matches[[
                "patient_id",
                "age_group",
                "sex",
                "rurality",
                "chronic_condition_count",
                "ed_escalation_risk_level",
                "readmission_vulnerability_level",
                "avoidable_utilization_risk_level",
                "care_complexity_level",
                "top_risk_driver",
                "recommended_care_route",
                "route_explanation",
                "equity_warning",
                "similarity_score"
            ]].head(10)
        )

# Connect the button to the function
match_button.on_click(on_match_clicked)

# ____________________________________________________________
# 5. Display the input form
# ____________________________________________________________
display(Markdown("## Hypothetical Patient Input"))
display(demo_age_group)
display(demo_sex)
display(demo_rurality)
display(demo_diabetes)
display(demo_hypertension)
display(demo_copd)
display(demo_heart)
display(demo_mental)
display(demo_digital)
display(demo_primary_care)
display(demo_mobility)
display(demo_transport)
display(demo_n_matches)
display(match_button)
display(match_output)


## Hypothetical Patient Input

Dropdown(description='Age Group:', layout=Layout(width='350px'), options=('0 to 4 years', '10 to 14 years', '1…

Dropdown(description='Sex:', layout=Layout(width='250px'), options=('Female', 'Male'), style=DescriptionStyle(…

Dropdown(description='Rurality:', layout=Layout(width='250px'), options=('Urban', 'Rural'), style=DescriptionS…

Dropdown(description='Diabetes:', layout=Layout(width='250px'), options=(0, 1), style=DescriptionStyle(descrip…

...

Dropdown(description='Mobility Limitation:', layout=Layout(width='300px'), options=(0, 1), style=DescriptionSt…

Dropdown(description='Transport Barrier:', layout=Layout(width='280px'), options=(0, 1), style=DescriptionStyl…

IntSlider(value=25, description='Top Matches:', layout=Layout(width='400px'), min=5, step=5, style=SliderStyle…

Button(button_style='success', description='Find Similar Synthetic Patients', icon='search', style=ButtonStyle…

Output()

### What this section does
This step introduces an **interactive demonstration layer** for the system.

The user can enter a **hypothetical patient profile**, and the system will search the synthetic population to find **patients with similar characteristics**.

From those matches, the system summarizes:

- **Likely risk pattern**
- **Likely care routing pathway**
- **Most common risk driver**

This provides an example of how the synthetic system could support **case-level navigation insights**.

### Why this step exists
This feature adds a **live demonstration component** to the project.

Instead of only presenting population-level simulations, it allows judges to see **how the system behaves when evaluating an individual patient scenario**, while still relying entirely on **synthetic data**.

This makes the project feel more:

- **Interactive**
- **Practical**
- **Decision-support oriented**

### Technologies used
This stage uses lightweight matching and summarization logic, including:

- **Similarity-based patient matching**
- **Feature comparison across synthetic records**
- **Aggregation of matched patient outcomes**
- **Summary generation for risk patterns and care pathways**

The approach is intentionally simple and explainable, making it easy to demonstrate during a presentation.

### Why this matters
This step adds a **second level of insight** to the system.

The earlier stages provide **population-level simulations**, while this section introduces **patient-level illustrative matching**.

Together, they create a system that supports both:

- **Strategic healthcare planning**
- **Illustrative patient navigation scenarios**

This combination strengthens the overall demonstration of the project.

**This input block demonstrates a hypothetical patient scenario.**  
The system compares a user-entered patient profile to similar synthetic patients and shows the most likely risk pattern and care pathway.

### Important note
This feature is designed as a **demonstration tool**, not a clinical decision system.  
It illustrates how synthetic population insights could support navigation and planning without using real patient data.